In [2]:

!pip install psycopg2-binary sqlalchemy -q

import os
import pandas as pd
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score


url = "https://raw.githubusercontent.com/pplonski/datasets-for-start/refs/heads/master/employee_attrition/HR-Employee-Attrition-All.csv"
df = pd.read_csv(url)

df['Attrition'] = df['Attrition'].map({'No': 0, 'Yes': 1})
df['OverTime'] = df['OverTime'].map({'No': 0, 'Yes': 1})
df = pd.get_dummies(df, columns=['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus'], drop_first=True)
df = df.drop(columns=['EmployeeCount', 'StandardHours', 'EmployeeNumber', 'Over18'])

print("Cleaned data shape:", df.shape)

Cleaned data shape: (1470, 45)


In [3]:

os.environ['NEON_CONNECTION_STRING'] = "your-neon-connection-string-here"

conn_string = os.environ.get('NEON_CONNECTION_STRING')
conn_string = conn_string.replace("postgresql://", "postgresql+psycopg2://")
engine = create_engine(conn_string)


df.to_sql('employees', engine, if_exists='replace', index=False)
print("uploaded")


df_from_db = pd.read_sql("SELECT * FROM employees", engine)
print("From DB:", df_from_db.shape)

uploaded
From DB: (1470, 45)


In [4]:

X = df_from_db.drop(columns=['Attrition'])
y = df_from_db['Attrition']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


log_reg = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced').fit(X_train, y_train)
nb = GaussianNB().fit(X_train, y_train)


models = {'Logistic Regression': (log_reg, X_test_s), 'Random Forest': (rf, X_test), 'Naive Bayes': (nb, X_test)}

results = pd.DataFrame([
    {
        'Model': name,
        'Accuracy': accuracy_score(y_test, m.predict(Xt)),
        'Precision (leavers)': precision_score(y_test, m.predict(Xt)),
        'Recall (leavers)': recall_score(y_test, m.predict(Xt)),
        'F1 (leavers)': f1_score(y_test, m.predict(Xt)),
    }
    for name, (m, Xt) in models.items()
])
print(results.round(3))

                 Model  Accuracy  Precision (leavers)  Recall (leavers)  \
0  Logistic Regression     0.861                0.615             0.340   
1        Random Forest     0.840                0.500             0.085   
2          Naive Bayes     0.721                0.312             0.617   

   F1 (leavers)  
0         0.438  
1         0.145  
2         0.414  
